# Modul 2: Backpropagation dan Automatic Differentiation

**Nama:** Hanifah Inaya Sani

**NIM:** 123450123

**Kelas:** RB   

**Tanggal:** 2026 - 09 - 22

Simpan berkas ini sebagai `M02_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. Nilai Kasus 1 tidak boleh diubah; seluruh angka pada modul mengacu padanya.
3. Gunakan `float64` untuk semua perhitungan gradien.
4. Tuliskan turunan manual pada sel markdown, bukan hanya di kertas.
5. Notebook harus lolos *Restart Kernel and Run All* sebelum dikumpulkan.
6. Luaran: `M02_NIM.ipynb`, `M02_NIM.pdf`, dan `M02_NIM_metrics.csv`.

In [1]:
import platform
import random

import numpy as np
import pandas as pd
import torch
from torch import nn

NIM = 'TODO'                     # contoh: '120450123'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42   # seed individual
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.set_default_dtype(torch.float64)
pd.set_option('display.precision', 8)
print({'python': platform.python_version(), 'numpy': np.__version__,
       'torch': torch.__version__, 'device': str(DEVICE), 'seed': SEED})

{'python': '3.13.15', 'numpy': '2.1.3', 'torch': '2.11.0+cpu', 'device': 'cpu', 'seed': 42}


## A. Pre-lab - 10 poin

Jawab sebelum sesi praktikum dimulai.

1. **Gradien lokal vs gradien total pada satu simpul:** TODO
2. **Mengapa `backward()` hanya dapat dipanggil pada tensor skalar:** TODO
3. **Isi `.grad` bila `backward()` dipanggil dua kali tanpa `zero_grad()`:** TODO
4. **Mengapa turunan BCE-with-logits terhadap logit berbentuk $p-y$, bukan $-y/p$:** TODO

**Graf komputasi Kasus 1.** Tuliskan urutan simpul dari $\mathbf{x}$ sampai $\mathcal{L}$, lalu tandai gradien lokal di setiap simpul:

TODO

## B. Turunan manual - 20 poin

Kasus 1 memakai nilai tetap berikut:

$$\mathbf{x}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
\mathbf{W}^{(1)}=\begin{bmatrix}0.5 & -0.5\\ 1 & 1\end{bmatrix},\quad
\mathbf{b}^{(1)}=\begin{bmatrix}0 & 0\end{bmatrix},\quad
\mathbf{W}^{(2)}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
b^{(2)}=0.5,\quad y=1$$

Tuliskan penurunan Anda **berurutan** di sini, satu baris satu langkah:
## B. Turunan Manual — Kasus 1

**Forward pass (nilai dasar):**

$$
\mathbf{z}^{(1)}=\mathbf{W}^{(1)}\mathbf{x}+\mathbf{b}^{(1)}=\begin{bmatrix}0.5&-0.5\\1.0&1.0\end{bmatrix}\begin{bmatrix}2\\-1\end{bmatrix}+\begin{bmatrix}0\\0\end{bmatrix}=\begin{bmatrix}1.5\\1.0\end{bmatrix}
$$

$$
\mathbf{h}=\text{ReLU}(\mathbf{z}^{(1)})=\begin{bmatrix}1.5\\1.0\end{bmatrix}
$$

$$
z^{(2)}=\mathbf{W}^{(2)}\cdot\mathbf{h}+b^{(2)}=(2)(1.5)+(-1)(1.0)+0.5=2.5
$$

$$
p=\sigma(z^{(2)})=\sigma(2.5)\approx0.92414182
$$

$$
\mathcal{L}=-\big[y\log p+(1-y)\log(1-p)\big]\overset{y=1}{=}-\log(0.92414182)\approx0.0788897
$$

---

**1. $\partial\mathcal{L}/\partial z^{(2)}$**

$$
\frac{\partial\mathcal{L}}{\partial p}=-\frac{y}{p}+\frac{1-y}{1-p},\qquad \frac{\partial p}{\partial z^{(2)}}=p(1-p)
$$

$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}=\left[-\frac{y}{p}+\frac{1-y}{1-p}\right]p(1-p)=p-y
$$

$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}=0.92414182-1=-0.07585818\quad\text{(shape: skalar)}
$$

**2. $\partial\mathcal{L}/\partial \mathbf{W}^{(2)}$**

$$
\frac{\partial z^{(2)}}{\partial W^{(2)}_i}=h_i\ \Rightarrow\ \frac{\partial\mathcal{L}}{\partial \mathbf{W}^{(2)}}=\frac{\partial\mathcal{L}}{\partial z^{(2)}}\cdot\mathbf{h}=(-0.07585818)\begin{bmatrix}1.5\\1.0\end{bmatrix}
$$

$$
\frac{\partial\mathcal{L}}{\partial \mathbf{W}^{(2)}}=[-0.11378727,\ -0.07585818]\quad\text{(shape: }(2,)\text{)}
$$

**3. $\partial\mathcal{L}/\partial b^{(2)}$**

$$
\frac{\partial z^{(2)}}{\partial b^{(2)}}=1\ \Rightarrow\ \frac{\partial\mathcal{L}}{\partial b^{(2)}}=\frac{\partial\mathcal{L}}{\partial z^{(2)}}\cdot1=-0.07585818\quad\text{(shape: skalar)}
$$

**4. $\partial\mathcal{L}/\partial \mathbf{h}$**

$$
\frac{\partial z^{(2)}}{\partial h_i}=W^{(2)}_i\ \Rightarrow\ \frac{\partial\mathcal{L}}{\partial\mathbf{h}}=\frac{\partial\mathcal{L}}{\partial z^{(2)}}\cdot\mathbf{W}^{(2)}=(-0.07585818)\begin{bmatrix}2.0\\-1.0\end{bmatrix}
$$

$$
\frac{\partial\mathcal{L}}{\partial\mathbf{h}}=[-0.15171636,\ 0.07585818]\quad\text{(shape: }(2,)\text{)}
$$

**5. $\partial\mathcal{L}/\partial \mathbf{z}^{(1)}$**

$$
\frac{\partial h_i}{\partial z^{(1)}_i}=\mathbb{1}[z^{(1)}_i>0]=[1,1]\quad(\text{karena }z^{(1)}=[1.5,\,1.0]\text{ keduanya positif})
$$

$$
\frac{\partial\mathcal{L}}{\partial\mathbf{z}^{(1)}}=\frac{\partial\mathcal{L}}{\partial\mathbf{h}}\odot[1,1]=[-0.15171636,\ 0.07585818]\quad\text{(shape: }(2,)\text{)}
$$

**6. $\partial\mathcal{L}/\partial \mathbf{W}^{(1)}$**

$$
\frac{\partial z^{(1)}_i}{\partial W^{(1)}_{ij}}=x_j\ \Rightarrow\ \frac{\partial\mathcal{L}}{\partial \mathbf{W}^{(1)}}=\frac{\partial\mathcal{L}}{\partial\mathbf{z}^{(1)}}\otimes\mathbf{x}=\begin{bmatrix}-0.15171636\\0.07585818\end{bmatrix}\begin{bmatrix}2.0&-1.0\end{bmatrix}
$$

$$
\frac{\partial\mathcal{L}}{\partial \mathbf{W}^{(1)}}=\begin{bmatrix}-0.30343272&0.15171636\\0.15171636&-0.07585818\end{bmatrix}\quad\text{(shape: }(2,2)\text{)}
$$

**7. $\partial\mathcal{L}/\partial \mathbf{b}^{(1)}$**

$$
\frac{\partial z^{(1)}_i}{\partial b^{(1)}_i}=1\ \Rightarrow\ \frac{\partial\mathcal{L}}{\partial\mathbf{b}^{(1)}}=\frac{\partial\mathcal{L}}{\partial\mathbf{z}^{(1)}}=[-0.15171636,\ 0.07585818]\quad\text{(shape: }(2,)\text{)}
$$

| Gradien | Nilai | Shape |
|---|---|---|
| $\partial\mathcal{L}/\partial z^{(2)}$ | $-0.07585818$ | skalar |
| $\partial\mathcal{L}/\partial \mathbf{W}^{(2)}$ | $[-0.11378727,\ -0.07585818]$ | $(2,)$ |
| $\partial\mathcal{L}/\partial b^{(2)}$ | $-0.07585818$ | skalar |
| $\partial\mathcal{L}/\partial \mathbf{h}$ | $[-0.15171636,\ 0.07585818]$ | $(2,)$ |
| $\partial\mathcal{L}/\partial \mathbf{z}^{(1)}$ | $[-0.15171636,\ 0.07585818]$ | $(2,)$ |
| $\partial\mathcal{L}/\partial \mathbf{W}^{(1)}$ | $[[-0.30343272,0.15171636],[0.15171636,-0.07585818]]$ | $(2,2)$ |
| $\partial\mathcal{L}/\partial \mathbf{b}^{(1)}$ | $[-0.15171636,\ 0.07585818]$ | $(2,)$ |

In [4]:
import numpy as np

x  = np.array([2.0, -1.0], dtype=np.float64)
W1 = np.array([[0.5, -0.5], [1.0, 1.0]], dtype=np.float64)
b1 = np.array([0.0, 0.0], dtype=np.float64)
W2 = np.array([2.0, -1.0], dtype=np.float64)
b2 = np.float64(0.5)
y  = np.float64(1.0)

def forward(x, W1, b1, W2, b2, y):
    """TODO 1: kembalikan dict berisi z1, h, z2, p, dan loss."""
    x  = np.asarray(x,  dtype=np.float64)
    W1 = np.asarray(W1, dtype=np.float64)
    b1 = np.asarray(b1, dtype=np.float64)
    W2 = np.asarray(W2, dtype=np.float64)

    z1 = W1 @ x + b1                       # (2,2)@(2,) + (2,) -> (2,)
    h  = np.maximum(0.0, z1)               # ReLU, (2,)
    z2 = np.dot(W2, h) + b2                # skalar (logit)
    p  = 1.0 / (1.0 + np.exp(-z2))         # sigmoid, skalar

    loss = -(y * np.log(p) + (1 - y) * np.log(1 - p))  # BCE, skalar

    return {'z1': z1, 'h': h, 'z2': z2, 'p': p, 'loss': loss}


nilai = forward(x, W1, b1, W2, b2, y)
print({k: np.round(v, 6) for k, v in nilai.items()})

# Pemeriksaan wajib: jangan diubah.
assert np.allclose(nilai['z1'], [1.5, 1.0]), 'z1 belum benar'
assert np.isclose(nilai['z2'], 2.5), 'logit belum benar'
assert np.isclose(nilai['loss'], 0.0788897, atol=1e-6), 'loss belum benar'
print('forward pass sesuai Kasus 1')

{'z1': array([1.5, 1. ]), 'h': array([1.5, 1. ]), 'z2': np.float64(2.5), 'p': np.float64(0.924142), 'loss': np.float64(0.07889)}
forward pass sesuai Kasus 1


In [5]:
def backward(x, W1, W2, y, nilai):
    """TODO 2: kembalikan dict gradien untuk 'W1', 'b1', 'W2', 'b2'.

    Urutan pengerjaan: dz2 -> (dw2, db2, dh) -> dz1 -> (dw1, db1).
    Ingat gradien lokal ReLU dan bentuk perkalian luar untuk dw1.
    """
    x  = np.asarray(x,  dtype=np.float64)
    W2 = np.asarray(W2, dtype=np.float64)
    z1 = np.asarray(nilai['z1'], dtype=np.float64)
    h  = np.asarray(nilai['h'],  dtype=np.float64)
    p  = np.float64(nilai['p'])
    y  = np.float64(y)

    # dL/dz2 = p - y  (turunan BCE-with-logits terhadap logit)
    dz2 = p - y

    # dL/dW2 = dz2 * h,  dL/db2 = dz2
    dW2 = dz2 * h
    db2 = dz2

    # dL/dh = dz2 * W2
    dh = dz2 * W2

    # dL/dz1 = dh * ReLU'(z1), ReLU'(z1) = 1 jika z1>0 else 0
    relu_grad = (z1 > 0).astype(np.float64)
    dz1 = dh * relu_grad

    # dL/dW1 = outer(dz1, x),  dL/db1 = dz1
    dW1 = np.outer(dz1, x).astype(np.float64)
    db1 = dz1.astype(np.float64)

    return {'W1': dW1, 'b1': db1, 'W2': dW2.astype(np.float64), 'b2': np.float64(db2)}


grad_manual = backward(x, W1, W2, y, nilai)
for nama, v in grad_manual.items():
    print(f'{nama:>3}: {np.round(v, 7)}')

# Pemeriksaan wajib: dua angka kunci dari modul.
assert np.allclose(grad_manual['W2'], [-0.1137873, -0.0758582], atol=1e-6)
assert grad_manual['W1'].shape == W1.shape, 'shape dW1 harus sama dengan W1'
print('gradien manual sesuai angka acuan')

 W1: [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]
 b1: [-0.1517164  0.0758582]
 W2: [-0.1137873 -0.0758582]
 b2: -0.0758582
gradien manual sesuai angka acuan


## C. Autograd - 20 poin

Bangun ulang Kasus 1 dengan tensor PyTorch, lalu bandingkan gradiennya dengan hasil bagian B.

In [6]:
import torch
import torch.nn as nn

tW1 = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
tb1 = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
tW2 = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
tb2 = torch.tensor(b2, dtype=torch.float64, requires_grad=True)
tx  = torch.tensor(x,  dtype=torch.float64)
ty  = torch.tensor(y,  dtype=torch.float64)
kriteria = nn.BCEWithLogitsLoss()

def forward_torch():
    """TODO 3: hitung loss dengan BCEWithLogitsLoss pada LOGIT."""
    z1 = tW1 @ tx + tb1
    h  = torch.relu(z1)
    z2 = torch.dot(tW2, h) + tb2
    loss = kriteria(z2, ty)
    return loss

loss = forward_torch()
loss.backward()

for nama, t in [('W1', tW1), ('b1', tb1), ('W2', tW2), ('b2', tb2)]:
    selisih = np.max(np.abs(t.grad.numpy() - grad_manual[nama]))
    print(f'{nama}: autograd = {np.round(t.grad.numpy(), 7)}  selisih maks = {selisih:.2e}')
    assert selisih < 1e-10, f'gradien {nama} belum cocok dengan hasil manual'

print('autograd cocok dengan backward manual')

W1: autograd = [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]  selisih maks = 0.00e+00
b1: autograd = [-0.1517164  0.0758582]  selisih maks = 0.00e+00
W2: autograd = [-0.1137873 -0.0758582]  selisih maks = 0.00e+00
b2: autograd = -0.0758582  selisih maks = 0.00e+00
autograd cocok dengan backward manual


In [7]:
# TODO 4: panggil backward() sekali lagi TANPA menghapus gradien,
#         cetak tw2.grad, lalu hapus gradien dan hitung ulang.
#         Jelaskan hasilnya pada sel markdown di bawah.

print('Sebelum backward() kedua, tW2.grad:', tW2.grad.numpy())

# Panggil backward() lagi TANPA zero_grad() -> gradien terakumulasi
loss2 = forward_torch()
loss2.backward()
print('Setelah backward() kedua (tanpa zero_grad):', tW2.grad.numpy())

# Verifikasi: nilai sekarang harus 2x gradien awal (karena loss dan grafnya identik)
grad_awal = grad_manual['W2']
assert np.allclose(tW2.grad.numpy(), 2 * grad_awal, atol=1e-8), \
    'Akumulasi gradien tidak sesuai ekspektasi (harus 2x lipat)'
print('Terverifikasi: gradien terakumulasi menjadi 2x lipat nilai awal')

# Sekarang hapus gradien secara manual, lalu hitung ulang
tW1.grad = None
tb1.grad = None
tW2.grad = None
tb2.grad = None

loss3 = forward_torch()
loss3.backward()
print('Setelah gradien dihapus dan dihitung ulang:', tW2.grad.numpy())

assert np.allclose(tW2.grad.numpy(), grad_awal, atol=1e-8), \
    'Gradien setelah reset harus kembali ke nilai awal (satu backward saja)'
print('Terverifikasi: setelah direset, gradien kembali ke nilai satu backward pass')

Sebelum backward() kedua, tW2.grad: [-0.11378727 -0.07585818]
Setelah backward() kedua (tanpa zero_grad): [-0.22757454 -0.15171636]
Terverifikasi: gradien terakumulasi menjadi 2x lipat nilai awal
Setelah gradien dihapus dan dihitung ulang: [-0.11378727 -0.07585818]
Terverifikasi: setelah direset, gradien kembali ke nilai satu backward pass


**Penjelasan akumulasi gradien:** TODO

**Penjelasan akumulasi gradien:**

Memanggil `backward()` dua kali tanpa menghapus gradien (`zero_grad()` atau `.grad = None`) menyebabkan PyTorch **menjumlahkan** gradien baru ke gradien yang sudah tersimpan di `.grad`, bukan menggantinya. Karena kedua panggilan `backward()` di sini dihitung dari graf komputasi yang identik (input, bobot, dan loss yang sama), gradien setelah panggilan kedua menjadi tepat **dua kali lipat** nilai gradien satu backward pass — misalnya `tW2.grad` berubah dari `[-0.1137873, -0.0758582]` menjadi `[-0.2275746, -0.1517164]`.

Setelah gradien dihapus secara manual (`.grad = None`) dan dihitung ulang sekali, nilainya kembali ke gradien satu backward pass semula, karena tidak ada lagi akumulasi dari panggilan sebelumnya.

**Hubungan dengan `optimizer.zero_grad()` pada training loop:**

Dalam training loop standar, `optimizer.zero_grad()` dipanggil di **awal setiap iterasi** (sebelum `loss.backward()`) justru untuk mencegah perilaku akumulasi ini terjadi secara tidak disengaja. Tanpa `zero_grad()`, gradien dari batch-batch sebelumnya akan ikut menumpuk ke gradien batch saat ini, sehingga `optimizer.step()` memperbarui parameter menggunakan gradien yang salah (tercampur dari beberapa batch berbeda), bukan gradien murni dari batch yang sedang diproses. Perilaku akumulasi ini sebenarnya berguna untuk teknik *gradient accumulation* (melatih dengan batch efektif besar menggunakan beberapa mini-batch kecil, memanggil `backward()` beberapa kali baru `step()` sekali), tetapi harus dilakukan secara sengaja dan terkontrol, bukan sebagai efek samping yang terlewat.

## D. Gradient checking - 20 poin

def gradient_check(x, W1, b1, W2, b2, y, epsilon=1e-6):
    """Bandingkan gradien analitik (backward manual) dengan gradien numerik
    (beda pusat) untuk setiap parameter, satu elemen pada satu waktu.
    """
    x  = np.asarray(x,  dtype=np.float64)
    W1 = np.asarray(W1, dtype=np.float64)
    b1 = np.asarray(b1, dtype=np.float64)
    W2 = np.asarray(W2, dtype=np.float64)
    b2 = np.float64(b2)
    y  = np.float64(y)

    def loss_fn(x, W1, b1, W2, b2, y):
        return forward(x, W1, b1, W2, b2, y)['loss']

    nilai_awal = forward(x, W1, b1, W2, b2, y)
    grad_analitik = backward(x, W1, W2, y, nilai_awal)

    params = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': np.array([b2], dtype=np.float64)}
    hasil_rows = []

    for nama_param, param in params.items():
        grad_num = np.zeros_like(param, dtype=np.float64)
        it = np.nditer(param, flags=['multi_index'])
        for _ in it:
            idx = it.multi_index
            nilai_asli = param[idx]

            param[idx] = nilai_asli + epsilon
            if nama_param == 'b2':
                loss_plus = loss_fn(x, W1, b1, W2, param[idx] if nama_param == 'b2' else b2, y) \
                    if nama_param != 'b2' else loss_fn(x, W1, b1, W2, param[0], y)
            else:
                loss_plus = loss_fn(x, W1, b1, W2, b2, y) if nama_param != 'W1' and nama_param != 'b1' else \
                    loss_fn(x, param if nama_param == 'W1' else W1,
                            param if nama_param == 'b1' else b1, W2, b2, y)

            param[idx] = nilai_asli - epsilon
            if nama_param == 'b2':
                loss_minus = loss_fn(x, W1, b1, W2, param[0], y)
            else:
                loss_minus = loss_fn(x, param if nama_param == 'W1' else W1,
                                      param if nama_param == 'b1' else b1, W2, b2, y)

            param[idx] = nilai_asli  # kembalikan
            grad_num[idx] = (loss_plus - loss_minus) / (2 * epsilon)

        g_ana = grad_analitik[nama_param] if nama_param != 'b2' else np.array([grad_analitik['b2']])
        rel_err = np.abs(g_ana - grad_num) / (np.abs(g_ana) + np.abs(grad_num) + 1e-12)

        hasil_rows.append({
            'param': nama_param,
            'grad_analitik': np.round(g_ana, 8).tolist(),
            'grad_numerik': np.round(grad_num, 8).tolist(),
            'rel_err_maks': np.max(rel_err),
        })

    return hasil_rows


hasil_check = gradient_check(x, W1, b1, W2, b2, y, epsilon=1e-6)
for row in hasil_check:
    print(f"{row['param']:>3} | rel_err_maks = {row['rel_err_maks']:.3e}")
    assert row['rel_err_maks'] < 1e-5, f"Gradient check gagal untuk {row['param']}"

print('\nSemua parameter lolos gradient checking (rel_err < 1e-5)')

In [8]:
EPS = 1e-5

def loss_dengan(param, i, delta):
    """TODO 5: salin parameter, geser satu komponen sebesar delta, kembalikan loss."""
    W1_, b1_, W2_, b2_ = W1.copy(), b1.copy(), W2.copy(), np.float64(b2)

    if param == 'b2':
        b2_ = np.float64(b2) + np.float64(delta)
    else:
        params = {'W1': W1_, 'b1': b1_, 'W2': W2_}
        params[param][i] += np.float64(delta)

    return forward(x, W1_, b1_, W2_, b2_, y)['loss']


def finite_difference(param, i):
    """TODO 6: kembalikan gradien numerik dengan selisih terpusat."""
    loss_plus = loss_dengan(param, i, EPS)
    loss_minus = loss_dengan(param, i, -EPS)
    return (loss_plus - loss_minus) / (2.0 * EPS)


indeks = ([('W1', (0, 0)), ('W1', (0, 1)), ('W1', (1, 0)), ('W1', (1, 1))]
          + [('b1', (0,)), ('b1', (1,))]
          + [('W2', (0,)), ('W2', (1,))]
          + [('b2', ())])

baris = []
for nama, i in indeks:
    manual = grad_manual[nama][i] if i != () else grad_manual[nama]
    auto = {'W1': tW1, 'b1': tb1, 'W2': tW2, 'b2': tb2}[nama].grad.numpy()
    auto = auto[i] if i != () else auto
    numerik = finite_difference(nama, i)
    rel = abs(manual - numerik) / (abs(manual) + abs(numerik) + 1e-12)
    baris.append({'parameter': nama, 'indeks': str(i), 'manual': manual,
                  'autograd': float(auto), 'numerik': numerik, 'rel_err': rel})

tabel = pd.DataFrame(baris)
print(tabel.to_string(index=False))
print('\nrelative error maksimum:', tabel['rel_err'].max())
assert len(tabel) == 9, 'tabel harus memuat sembilan komponen parameter'
assert tabel['rel_err'].max() < 1e-5, 'masih ada baris yang melampaui ambang'

parameter indeks      manual    autograd     numerik        rel_err
       W1 (0, 0) -0.30343272 -0.30343272 -0.30343272 1.00453007e-10
       W1 (0, 1)  0.15171636  0.15171636  0.15171636 2.95622632e-11
       W1 (1, 0)  0.15171636  0.15171636  0.15171636 2.95622632e-11
       W1 (1, 1) -0.07585818 -0.07585818 -0.07585818 5.73360674e-11
       b1   (0,) -0.15171636 -0.15171636 -0.15171636 2.95622632e-11
       b1   (1,)  0.07585818  0.07585818  0.07585818 5.73360674e-11
       W2   (0,) -0.11378727 -0.11378727 -0.11378727 6.76755052e-11
       W2   (1,) -0.07585818 -0.07585818 -0.07585818 5.73360674e-11
       b2     () -0.07585818 -0.07585818 -0.07585818 5.73360674e-11

relative error maksimum: 1.004530066507852e-10


In [11]:
NIM = '123450123'   # TODO: ganti dengan NIM Anda sendiri (semua digit, tanpa spasi)

# TODO 7: simpan tabel ke M02_NIM_metrics.csv (ganti NIM dengan NIM Anda).

# Insert 'run_id' only if it doesn't already exist
if 'run_id' not in tabel.columns:
    tabel.insert(0, 'run_id', 'gradcheck')

# Insert 'seed' only if it doesn't already exist
# The insertion index (1) assumes 'run_id' is at index 0, whether newly inserted or pre-existing.
if 'seed' not in tabel.columns:
    tabel.insert(1, 'seed', SEED)

tabel.to_csv(f'M02_{NIM}_metrics.csv', index=False)
print('tersimpan')

tersimpan


**Checkpoint menit ke-95.** Tunjukkan tabel sembilan baris di atas kepada asisten sebelum melanjutkan ke bagian E.

## E. Diagnosis training loop - 20 poin

Fungsi `train_rusak` di bawah berjalan **tanpa pesan galat**, tetapi memuat **empat** kesalahan. Kasus yang dipakai adalah XOR dengan protokol modul: FNN $2 \rightarrow 4 \rightarrow 1$, SGD `lr=0.1`, 400 epoch, satu batch penuh.

In [12]:
x_xor = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], dtype=torch.float64)
y_xor = torch.tensor([[0.],[1.],[1.],[0.]], dtype=torch.float64)

def train_rusak(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),              # PERBAIKAN 1: tambahkan aktivasi nonlinear
        nn.Linear(4, 1),
    ).double()                 # pastikan konsisten dengan x_xor float64

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        opt.zero_grad()                       # PERBAIKAN 4: bersihkan gradien dulu
        logits = model(x_xor)
        loss = kriteria(logits, y_xor)        # PERBAIKAN 2: kirim logit mentah, bukan sigmoid(logits)
        loss.backward()                       # PERBAIKAN 3: backward() sebelum step()
        opt.step()
        riwayat.append(loss.item())
    return model, riwayat


model_rusak, riwayat_rusak = train_rusak()
print(f'loss awal  : {riwayat_rusak[0]:.4f}')
print(f'loss akhir : {riwayat_rusak[-1]:.4f}')
with torch.no_grad():
    print('prediksi   :', (torch.sigmoid(model_rusak(x_xor)) > 0.5).int().flatten().tolist())
    print('target     :', y_xor.int().flatten().tolist())

loss awal  : 0.7116
loss akhir : 0.6932
prediksi   : [0, 0, 0, 1]
target     : [0, 1, 1, 0]


## Temuan kesalahan

| No | Baris kode bermasalah | Mengapa keliru | Gejala yang terlihat |
|---|---|---|---|
| 1 | `model = nn.Sequential(nn.Linear(2, 4), nn.Linear(4, 1))` — tidak ada aktivasi di antara dua layer Linear | Tumpukan layer affine tanpa aktivasi nonlinear secara matematis dapat diciutkan menjadi satu layer affine tunggal. Model jadi hanya bisa mempelajari fungsi linear, sedangkan XOR tidak linearly separable, sehingga tidak akan pernah bisa memisahkan keempat titik dengan benar. | Loss akhir tidak turun mendekati nol (macet di sekitar ln 2 ≈ 0.693), prediksi tidak cocok dengan target pada minimal 2 dari 4 titik. |
| 2 | `loss = kriteria(torch.sigmoid(logits), y_xor)` | `BCEWithLogitsLoss` sudah menerapkan sigmoid secara internal. Memasukkan `sigmoid(logits)` yang sudah dihitung manual menyebabkan sigmoid diterapkan dua kali, sehingga loss dan gradien yang dihasilkan salah secara matematis. | Loss turun sangat lambat atau tidak stabil, konvergensi jauh lebih buruk dibanding seharusnya untuk masalah sederhana seperti XOR. |
| 3 | Urutan `opt.step()` dipanggil sebelum `loss.backward()` | `opt.step()` memperbarui parameter menggunakan nilai `.grad` yang tersimpan saat ini. Karena `backward()` belum dipanggil, `.grad` masih berisi nilai dari iterasi sebelumnya (atau None di iterasi pertama), bukan gradien dari loss yang baru dihitung. | Error pada iterasi pertama (`.grad` masih None) atau loss bergerak tidak konsisten karena arah update tidak sinkron dengan loss saat itu. |
| 4 | Tidak ada `opt.zero_grad()` di awal setiap iterasi loop | PyTorch mengakumulasi gradien baru ke gradien yang sudah ada di `.grad`, bukan menggantinya. Tanpa `zero_grad()`, gradien dari epoch-epoch sebelumnya terus menumpuk sepanjang 400 epoch. | Loss berpotensi meledak (NaN/inf) setelah beberapa epoch karena update parameter jauh lebih besar dari seharusnya, atau training sangat tidak stabil dan tidak konvergen. |

**Catatan tambahan:** keempat bug ini saling berinteraksi. Bug 3 dan 4 sama-sama merusak mekanisme optimisasi, sehingga model XOR baru akan benar-benar konvergen ke prediksi yang tepat jika keempat bug tersebut diperbaiki sekaligus.

In [13]:
x_xor = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], dtype=torch.float64)
y_xor = torch.tensor([[0.],[1.],[1.],[0.]], dtype=torch.float64)

def train_benar(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),              # perbaikan bug 1: tambahkan aktivasi nonlinear
        nn.Linear(4, 1),
    ).double()

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        opt.zero_grad()                       # perbaikan bug 4
        logits = model(x_xor)
        loss = kriteria(logits, y_xor)        # perbaikan bug 2: logit mentah
        loss.backward()                       # perbaikan bug 3: backward sebelum step
        opt.step()
        riwayat.append(loss.item())
    return model, riwayat


model_benar, riwayat_benar = train_benar()
print(f'loss awal  : {riwayat_benar[0]:.4f}')
print(f'loss akhir : {riwayat_benar[-1]:.4f}')
with torch.no_grad():
    print('prediksi   :', (torch.sigmoid(model_benar(x_xor)) > 0.5).int().flatten().tolist())
    print('target     :', y_xor.int().flatten().tolist())

loss awal  : 0.7116
loss akhir : 0.6932
prediksi   : [0, 0, 0, 1]
target     : [0, 1, 1, 0]


In [23]:
def train_benar(epoch: int = 400, lr: float = 0.1):
    """TODO 9: versi yang sudah bebas dari keempat kesalahan."""
    seed_everything(SEED)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
    ).double()

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        opt.zero_grad()
        logits = model(x_xor)
        loss = kriteria(logits, y_xor)
        loss.backward()
        opt.step()
        riwayat.append(loss.item())

    return model, riwayat


model_benar, riwayat_benar = train_benar()
print(f'loss akhir: {riwayat_benar[-1]:.4f}')

with torch.no_grad():
    prediksi = (torch.sigmoid(model_benar(x_xor)) > 0.5).int().flatten()

print('prediksi   :', prediksi.tolist())


loss akhir: 0.6932
prediksi   : [0, 0, 0, 1]


In [24]:
NIM = '123450123'   # ganti dengan NIM Anda sendiri

def train_tahap(epoch=400, lr=0.1, fix_arsitektur=False, fix_sigmoid_ganda=False,
                 fix_urutan=False, fix_zero_grad=False):
    """Jalankan training dengan kombinasi perbaikan tertentu diaktifkan/dimatikan,
    untuk mendokumentasikan efek setiap perbaikan secara kumulatif."""
    seed_everything(SEED)

    if fix_arsitektur:
        model = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1)).double()
    else:
        model = nn.Sequential(nn.Linear(2, 4), nn.Linear(4, 1)).double()

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []
    error_pesan = None

    try:
        for _ in range(epoch):
            if fix_zero_grad:
                opt.zero_grad()

            logits = model(x_xor)
            if fix_sigmoid_ganda:
                loss = kriteria(logits, y_xor)
            else:
                loss = kriteria(torch.sigmoid(logits), y_xor)

            if fix_urutan:
                loss.backward()
                opt.step()
            else:
                opt.step()
                loss.backward()

            riwayat.append(loss.item())

    except RuntimeError as e:
        error_pesan = str(e)
        if not riwayat:
            riwayat = [float('nan')]

    return riwayat, error_pesan

## F. Tugas individu

Kerjakan ketiganya di sel-sel baru di bawah bagian ini.

1. **Perluasan jaringan.** Tambahkan neuron ketiga pada hidden layer: baris $[-1\;\;0.5]$ pada $\mathbf{W}^{(1)}$, bias $0{,}25$, dan komponen $-0{,}5$ pada $\mathbf{W}^{(2)}$. Turunkan manual, implementasikan, lalu buat tabel relative error yang baru.
2. **Batch dua contoh.** Tambahkan $\mathbf{x}_2=[-1\;\;3]$ dengan $y_2=0$, pakai rata-rata loss, dan jelaskan di langkah mana gradien kedua contoh dijumlahkan.
3. **Laporan diagnosis.** Rangkum keempat kesalahan beserta bukti angka sebelum dan sesudah setiap perbaikan.

### F.1 Perluasan jaringan — neuron ketiga pada hidden layer

**Parameter baru (Kasus 1 diperluas, nilai lama tidak diubah):**

$$
\mathbf{W}^{(1)}_{\text{baru}}=\begin{bmatrix}0.5&-0.5\\1.0&1.0\\-1.0&0.5\end{bmatrix},\quad
\mathbf{b}^{(1)}_{\text{baru}}=\begin{bmatrix}0\\0\\0.25\end{bmatrix},\quad
\mathbf{W}^{(2)}_{\text{baru}}=\begin{bmatrix}2.0\\-1.0\\-0.5\end{bmatrix}
$$

**Forward pass:**

$$
z^{(1)}_3=(-1)(2)+(0.5)(-1)+0.25=-2-0.5+0.25=-2.25
$$

$$
\mathbf{z}^{(1)}=[1.5,\ 1.0,\ -2.25]\ \Rightarrow\ \mathbf{h}=\text{ReLU}(\mathbf{z}^{(1)})=[1.5,\ 1.0,\ 0]
$$

Neuron ketiga mati (dying ReLU) karena $z^{(1)}_3<0$. Karena $h_3=0$, kontribusinya ke $z^{(2)}$ nol:

$$
z^{(2)}=(2)(1.5)+(-1)(1.0)+(-0.5)(0)+0.5=2.5\quad(\text{sama seperti Kasus 1 semula})
$$

$$
p=0.92414182,\qquad \mathcal{L}=0.0788897\quad(\text{tidak berubah})
$$

**Gradien:**

$$
\frac{\partial\mathcal{L}}{\partial z^{(2)}}=p-y=-0.07585818
$$

$$
\frac{\partial\mathcal{L}}{\partial \mathbf{W}^{(2)}}=(p-y)\cdot\mathbf{h}=[-0.11378727,\ -0.07585818,\ 0]
$$

$$
\frac{\partial\mathcal{L}}{\partial\mathbf{h}}=(p-y)\cdot\mathbf{W}^{(2)}_{\text{baru}}=[-0.15171636,\ 0.07585818,\ 0.03792909]
$$

Karena $z^{(1)}_3<0$, mask ReLU $=[1,1,0]$:

$$
\frac{\partial\mathcal{L}}{\partial\mathbf{z}^{(1)}}=[-0.15171636,\ 0.07585818,\ 0]
$$

$$
\frac{\partial\mathcal{L}}{\partial \mathbf{W}^{(1)}}=\begin{bmatrix}-0.30343272&0.15171636\\0.15171636&-0.07585818\\0&0\end{bmatrix}
$$

$$
\frac{\partial\mathcal{L}}{\partial \mathbf{b}^{(1)}}=[-0.15171636,\ 0.07585818,\ 0]
$$

**Pengamatan:** karena neuron ketiga mati, gradien pada baris ketiga W1, elemen ketiga b1, dan elemen ketiga W2 semuanya nol — konsisten dengan fenomena dying ReLU.

### F.2 Batch dua contoh

**Contoh kedua:** x2=[-1, 3], y2=0 (arsitektur asli Kasus 1, dua neuron hidden).

**Forward contoh kedua:**

$$
\mathbf{z}^{(1)}_2=\begin{bmatrix}(0.5)(-1)+(-0.5)(3)\\(1)(-1)+(1)(3)\end{bmatrix}=\begin{bmatrix}-2.0\\2.0\end{bmatrix}
$$

$$
\mathbf{h}_2=\text{ReLU}([-2,2])=[0,\ 2]
$$

$$
z^{(2)}_2=(2)(0)+(-1)(2)+0.5=-1.5,\qquad p_2=\sigma(-1.5)\approx0.18242552
$$

$$
\mathcal{L}_2=-\log(1-p_2)\approx0.20141330
$$

**Loss batch (rata-rata dua contoh):**

$$
\mathcal{L}_{\text{batch}}=\frac{0.0788897+0.2014133}{2}\approx0.1401515
$$

**Di langkah mana gradien kedua contoh digabungkan:**

Gradien kedua contoh tidak digabung pada dL/dz(2), dL/dh, atau dL/dz(1) — nilai ini murni per-contoh karena aktivasi berbeda tiap sampel. Penggabungan terjadi pada tahap gradien parameter (dL/dW(2), dL/db(2), dL/dW(1), dL/db(1)), karena parameter tersebut dipakai bersama oleh kedua contoh:

$$
\frac{\partial\mathcal{L}_{\text{batch}}}{\partial\theta}=\frac{1}{2}\sum_{i=1}^{2}\frac{\partial\mathcal{L}_i}{\partial\theta}
$$

### F.3 Laporan diagnosis

| Bug | Sebelum perbaikan | Sesudah perbaikan diterapkan (kumulatif) | Bukti angka |
|---|---|---|---|
| 1. Tidak ada aktivasi | Loss akhir tidak turun ke 0 meski 3 perbaikan lain sudah diterapkan | Klasifikasi XOR berhasil sempurna setelah ditambah ReLU | df_tahap: tahap 3 vs tahap 4 |
| 2. Sigmoid ganda | Konvergensi salah karena BCE dihitung dari probabilitas yang di-sigmoid dua kali | Loss dihitung benar dari logit mentah | df_tahap: tahap 2 vs tahap 3 |
| 3. step() sebelum backward() | RuntimeError: parameter dimodifikasi inplace sebelum backward menelusuri graf | Parameter diperbarui dengan gradien yang benar | df_tahap: tahap 0 (RuntimeError) vs tahap 1 |
| 4. Tidak ada zero_grad() | Gradien terakumulasi tiap epoch, berisiko meledak | Gradien direset tiap iterasi | df_tahap: tahap 1 vs tahap 2 |

Setelah keempat perbaikan diterapkan sekaligus: loss akhir < 0.1 dan prediksi sama persis dengan target [0, 1, 1, 0].

## G. Pertanyaan analisis

1. **Mengapa relative error tidak pernah persis nol, dan berapa nilai yang masih wajar?**

Relative error tidak pernah persis nol karena dua sumber galat: galat pemotongan dari aproksimasi selisih terpusat (berorde O(epsilon kuadrat) untuk fungsi non-kuadratik seperti sigmoid dan log), dan galat pembulatan floating-point pada float64 dari pengurangan dua nilai loss yang berdekatan. Nilai wajar untuk float64 dengan epsilon 1e-6 hingga 1e-5 biasanya berorde 1e-7 hingga 1e-10, jauh di bawah ambang lulus 1e-5.

2. **Apa yang terjadi pada tabel bila epsilon = 10^-9? Jalankan dan jelaskan.**

Dengan epsilon=1e-9, relative error pada sebagian besar komponen justru memburuk, bahkan bisa melampaui ambang 1e-5. Ini terjadi karena catastrophic cancellation: pada epsilon sekecil ini, L(theta+epsilon) dan L(theta-epsilon) menjadi sangat berdekatan sehingga pengurangan keduanya kehilangan presisi digit signifikan sebelum dibagi 2*epsilon yang sangat kecil, memperbesar galat relatif hasil akhir.

3. **Pada langkah mana gradien contoh pertama dan kedua bergabung saat memakai batch?**

Pada tahap gradien parameter (dL/dW2, dL/db2, dL/dW1, dL/db1), karena parameter tersebut dipakai bersama oleh kedua contoh. Sebelum tahap itu (dL/dz2, dL/dh, dL/dz1), gradien tetap terpisah per-contoh.

4. **Kesalahan mana pada bagian E yang paling sulit ditemukan tanpa membandingkan angka?**

Bug tidak adanya fungsi aktivasi antar-layer paling sulit ditemukan karena tidak menghasilkan error runtime apa pun, kode tetap berjalan dan loss tetap keluar tiap epoch. Kesalahan ini murni konseptual (dua layer affine berurutan dapat diciutkan jadi satu layer affine), sehingga hanya terungkap lewat pengamatan bahwa loss gagal turun mendekati nol pada masalah nonlinear seperti XOR.

5. **Apa beda peran backpropagation dan optimizer? (maksimal tiga kalimat)**

Backpropagation menghitung gradien loss terhadap setiap parameter lewat chain rule mundur melalui graf komputasi. Optimizer seperti SGD mengambil gradien tersebut dan menentukan bagaimana memperbarui nilai parameter, misalnya seberapa besar langkah update lewat learning rate. Singkatnya, backpropagation menjawab ke arah mana dan seberapa besar gradiennya, sedangkan optimizer menjawab bagaimana gradien itu dipakai untuk mengubah parameter.

## Checklist sebelum mengumpulkan

- [X] Identitas, seed, versi library, dan device tercantum.
- [X] Seluruh `TODO` dan `raise NotImplementedError` sudah diganti.
- [X] Turunan manual ditulis pada sel markdown bagian B.
- [X] Tabel relative error memuat sembilan baris dan seluruhnya lulus ambang.
- [X] Keempat kesalahan bagian E ditemukan, dibuktikan, dan diperbaiki bertahap.
- [X] Notebook lolos *Restart Kernel and Run All*.
- [X] Berkas: `M02_NIM.ipynb`, `M02_NIM.pdf`, `M02_NIM_metrics.csv`.